In [ ]:
import pennylane as qml
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
import numpy as np

In [ ]:
from lib.data_loader import *
from lib.U2_gate import *
from lib.U3_gate import *
from lib.U4_gate import *
from lib.train_test import *

In [ ]:
n_wires = 8
n_layers = 1 
dev = qml.device('default.qubit', wires=n_wires)

U2_1 = u2_1()
U2_2 = u2_2()
U3_CB_1 = u3_cb_1()
U3_CB_2 = u3_cb_2()
U3_AA_1 = u3_aa_1()
U4_AA_1 = u4_aa_1()
U4_CB_1 = u4_cb_1()
U4_CB_2 = u4_cb_2()
U4_NN_1 = u4_nn_1()
U4_NN_2 = u4_nn_2()

gate_list = [U2_1, U2_2, U3_CB_1, U3_CB_2, U3_AA_1, U4_AA_1, U4_CB_1, U4_CB_2, U4_NN_1, U4_NN_2]

In [ ]:
def createLayers(gate):
    def layer(weights, wires):
        n_wires = len(wires)
        for i in range(n_wires):
            tmp_wires = [(i + j) % n_wires for j in range(gate.gate_size)]
            gate.circuit(weights=weights[i], wires=tmp_wires)
    return layer

def create_circuit(dev, layer, n_layers, n_wires):
    @qml.qnode(dev)
    def circuit(inputs, weights):
        qml.AmplitudeEmbedding(inputs, wires=range(n_wires), pad_with=0.)
        qml.layer(layer, n_layers, weights, wires=range(n_wires))
        return qml.probs(wires=range(n_wires))
    return circuit

circuit_list = []

for gate in gate_list:
    layer = createLayers(gate)
    circuit = create_circuit(dev, layer, n_layers, n_wires)
    circuit_list.append(circuit)

qlayer_list = []

np.random.seed(42)

# qlayers
for i in range(len(gate_list)):
    shape = (n_layers, n_wires, gate_list[i].weight_size)
    weight_shapes = {"weights":shape}
    qlayer = qml.qnn.TorchLayer(circuit_list[i], weight_shapes)
    qlayer_list.append(qlayer)


In [ ]:
print(qlayer_list[len(qlayer_list)-1].weights)

In [ ]:
class QCNNModel(nn.Module):
    def __init__(self, qlayer):
        super().__init__()
        self.flatten = nn.Flatten()
        self.conv = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=2, stride=2, padding=2)  # 16*16
        self.relu = nn.ReLU()
        self.qlayer = qlayer
        self.fc2 = nn.Linear(256, 4)
    
    def forward(self, x):
        x = self.conv(x)      
        x = self.flatten(x)     
        x = qlayer(x)          
        x = self.relu(x)       
        x = self.fc2(x)         
        return x

In [ ]:
# MNIST
transform = transforms.Compose([transforms.ToTensor()])
train_dataset = datasets.MNIST(root="../data", train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root="../data", train=False, download=True, transform=transform)
train_loader, test_loader = getDataloader(train_dataset, test_dataset, range(4), train_size=3000, test_size=600, batch_size=25, shuffle=True)


In [ ]:

epochs = 30

for i in range(len(qlayer_list)):
    print(f"qlayer_list[{i+1}]\n-------------------------------\n-------------------------------")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = QCNNModel(qlayer_list[i]).to(device)
    print(model)
    loss_fn = nn.CrossEntropyLoss()
    learning_rate = 0.01
    optimizer = optim.Adam(model.parameters(), lr = learning_rate)
    for t in range(epochs):
        print(f"Epoch {t+1}\n-------------------------------")
        train(train_loader, model, loss_fn, 25, optimizer, device)
        test(test_loader, model, loss_fn, device)
print("Done!")